In [1]:
!pip install control
!pip install matplotlib
!pip install numpy
!pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 25.6 MB/s eta 0:00:00


#Exercício 1:
Uma estufa agrícola inteligente usa um aquecedor resistivo (PWM) e um sensor
DHT22 para manter a temperatura interna. Um teste em degrau aplicou Δu = 0,20
(20% de duty cycle) e a temperatura subiu de 22 °C até se estabilizar em 34 °C (ΔT
= 12 °C).
Pelo método de Smith, identificou-se na curva de reação:
* t₁ (28,3% do valor final) = 15 s

* t₂ (63,2% do valor final) = 42 s

a) Fazer a identificação da planta pelo método de Smith e encontrar a função
de transferência dela.

b) Calcular o valor de Kp, Ti e Td pelos métodos de Ziegler-Nichols (malha
aberta), CHR sem sobrevalor e IMC.

c) Plotar os gráficos de resposta a um degrau unitário.

d) Fazer uma análise comparativa dos métodos.



a) Identificação da planta pelo método de Smith

In [8]:
# Dados do problema
delta_u = 0.20  # Variação no duty cycle
delta_T = 12    # (34 - 22)
t1 = 15         # Tempo em s para 28.3% do valor final
t2 = 42         # Tempo em s para 63.2% do valor final

# 1. Função de transferência e o ganho do processo (K)
K = delta_T / delta_u
print(f"Ganho do Processo (K): {K:.2f} °C/(% duty cycle)")

# 2. Calcular a constante de tempo (tau) e o tempo morto (theta) pelo método de Smith (FOPDT)
# Para t1 (28.3% da resposta) e t2 (63.2% da resposta) após o tempo morto,
# temos aproximadamente as relações:
# theta + 0.3*tau = t1
# theta + tau = t2

tau = (t2 - t1) / 0.7
theta = t1 - 0.3 * tau

print(f"Constante de Tempo (tau): {tau:.2f} s")
print(f"Tempo Morto (theta): {theta:.2f} s")

# Função de Transferência da Planta (FOPDT):
# G(s) = K * e^(-theta*s) / (tau*s + 1)

Ganho do Processo (K): 60.00 °C/(% duty cycle)
Constante de Tempo (tau): 38.57 s
Tempo Morto (theta): 3.43 s


b) Cálculo de Kp, Ti e Td pelos métodos de Ziegler-Nichols (malha aberta), CHR sem sobrevalor e IMC.

In [5]:
# 1. Métodos de Ziegler-Nichols (Malha Aberta - PID)

Kp_ZN = 1.2 / (K * (theta / tau))
Ti_ZN = 2 * theta
Td_ZN = 0.5 * theta

print("--- Ziegler-Nichols (PID) ---")
print(f"Kp: {Kp_ZN:.4f}")
print(f"Ti: {Ti_ZN:.4f} s")
print(f"Td: {Td_ZN:.4f} s\n")

# 2. Métodos de CHR (sem sobrevalor - PID)

Kp_CHR = 0.95 / K * (tau / theta)
Ti_CHR = 2.4 * theta
Td_CHR = 0.4 * theta

print("--- CHR (sem sobrevalor - PID) ---")
print(f"Kp: {Kp_CHR:.4f}")
print(f"Ti: {Ti_CHR:.4f} s")
print(f"Td: {Td_CHR:.4f} s\n")

# 3. Métodos IMC (PID) - utilizando lambda = theta

lambda_imc = theta

Kp_IMC = (1 / K) * (tau + theta / 2) / (lambda_imc + theta / 2)
Ti_IMC = tau + theta / 2
Td_IMC = (tau * theta) / (2 * tau + theta)

print("--- IMC (PID com lambda = theta) ---")
print(f"Kp: {Kp_IMC:.4f}")
print(f"Ti: {Ti_IMC:.4f} s")
print(f"Td: {Td_IMC:.4f} s")

--- Ziegler-Nichols (PID) ---
Kp: 0.2250
Ti: 6.8571 s
Td: 1.7143 s

--- CHR (sem sobrevalor - PID) ---
Kp: 0.1781
Ti: 8.2286 s
Td: 1.3714 s

--- IMC (PID com lambda = theta) ---
Kp: 0.1306
Ti: 40.2857 s
Td: 1.6413 s


c) Plote do gráfico

In [6]:
import numpy as np
import control as cnt
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Parâmetros da planta identificados pelo método de Smith
k_plant = K # Usando o K calculado
tau_plant = tau # Usando o tau calculado
theta_plant = theta # Usando o theta calculado

# Função para simular e plotar a resposta ao degrau
def plot_step_response(Kp, Ti, Td, method_name, color, fig):
    # Função de Transferência da Planta (FOPDT): G(s) = K * e^(-theta*s) / (tau*s + 1)
    num_plant = np.array([k_plant])
    den_plant = np.array([tau_plant, 1])
    H_plant = cnt.tf(num_plant, den_plant)

    # Aproximação de Padé para o tempo morto
    n_pade = 10 # Ordem da aproximação de Padé
    (num_pade, den_pade) = cnt.pade(theta_plant, n_pade)
    H_pade = cnt.tf(num_pade, den_pade)

    # Controlador PID
    # Kp
    num_kp = np.array([Kp])
    den_kp = np.array([1])
    H_kp = cnt.tf(num_kp, den_kp)

    # Ki = Kp/Ti
    if Ti != 0: # Evitar divisão por zero para controladores P ou PD
        num_ki = np.array([Kp])
        den_ki = np.array([Ti, 0])
        H_ki = cnt.tf(num_ki, den_ki)
    else:
        H_ki = cnt.tf([0], [1]) # Termo integrativo zero se Ti for zero

    # Kd = Kp*Td
    num_kd = np.array([Kp * Td, 0])
    den_kd = np.array([1])
    H_kd = cnt.tf(num_kd, den_kd)

    # Controlador PID em paralelo
    H_ctrl_pi = cnt.parallel(H_kp, H_ki)
    H_ctrl_pid = cnt.parallel(H_ctrl_pi, H_kd)

    # Malha aberta: Planta + Atraso + Controlador
    H_open_loop_plant_delay = cnt.series(H_plant, H_pade)
    H_open_loop = cnt.series(H_open_loop_plant_delay, H_ctrl_pid)

    # Malha fechada
    H_cl = cnt.feedback(H_open_loop, 1)

    # Simulação da resposta ao degrau
    t = np.linspace(0, 150, 500) # Ajustar o tempo de simulação conforme necessário
    (t, y) = cnt.step_response(H_cl, t)

    # Adiciona a resposta ao gráfico
    fig.add_trace(go.Scatter(x=t, y=y, mode='lines', name=method_name, line=dict(color=color)))

    return fig

# Inicializa a figura Plotly
fig = go.Figure()

# Plotar para Ziegler-Nichols
fig = plot_step_response(Kp_ZN, Ti_ZN, Td_ZN, 'Ziegler-Nichols', 'blue', fig)

# Plotar para CHR sem sobrevalor
fig = plot_step_response(Kp_CHR, Ti_CHR, Td_CHR, 'CHR (sem sobrevalor)', 'red', fig)

# Plotar para IMC
fig = plot_step_response(Kp_IMC, Ti_IMC, Td_IMC, 'IMC', 'green', fig)

# Ajustes do layout do gráfico
fig.update_layout(
    title="Comparação das Respostas ao Degrau para Diferentes Métodos de Sintonia PID",
    xaxis_title="Tempo [s]",
    yaxis_title="Saída da Planta (°C)",
    width=900,
    height=600,
    template="plotly_white"
)

# Mostra o gráfico
fig.show()

# Exercício 2
Explique, em poucas linhas, por que os métodos de Ziegler-Nichols e CHR não
costumam ser usados como ajuste final de um controlador industrial, e sim como
ponto de partida.

**Resposta:**

Os métodos de Ziegler-Nichols e CHR são excelentes para obter um ponto de partida rápido e razoável para a sintonia de controladores PID. No entanto, raramente são usados como ajuste final em aplicações industriais pois:
*  não apresentarem robustez ideal, ou seja, o desempenho do controlador pode degradar-se se as características do processo mudarem;
* As regras de sintonia desses métodos visam um compromisso genérico entre estabilidade e tempo de resposta, mas nem sempre otimizam para requisitos específicos da aplicação;
* Considerações como ruído de medição, limites de atuação e não linearidades não são diretamente endereçadas por essas regras de sintonia simples.

#Exercício 3:
Para cada sistema abaixo, calcule Kp, Ti e Td para o controlador PID pelo
método de Ziegler-Nichols (malha aberta):

a) Irrigação automatizada.
Uma microbomba controla a umidade do solo medida por um sensor capacitivo
IoT. G(s) = 2,5·e^(−4s) / (20s+1).

b) Esteira transportadora industrial.
Um inversor de frequência com telemetria IoT controla a velocidade de uma
correia. G(s) = 1,8·e^(−0,8s) / (3s+1).

c) Compare os valores de Ti obtidos nos itens a) e b) com o tempo morto θ de cada
sistema. Em qual dos dois sistemas o controlador PID seria mais sensível a um
atraso adicional de comunicação Wi-Fi de 300 ms entre o sensor e o atuador?
Justifique

### Exercício 3: Cálculos dos parâmetros PID (Ziegler-Nichols - malha aberta)

In [12]:
# Parâmetros do método de Ziegler-Nichols (malha aberta) para PID:
# Kp = 1.2 / (K * (theta / tau))
# Ti = 2 * theta
# Td = 0.5 * theta

print("Irrigação automatizada")
# G(s) = 2,5·e^(−4s) / (20s+1)
K_a = 2.5
tau_a = 20
theta_a = 4

Kp_a = 1.2 / (K_a * (theta_a / tau_a))
Ti_a = 2 * theta_a
Td_a = 0.5 * theta_a

print(f"Kp (a): {Kp_a:.4f}")
print(f"Ti (a): {Ti_a:.4f} s")
print(f"Td (a): {Td_a:.4f} s\n")

print("Esteira transportadora industrial")
# G(s) = 1,8·e^(−0,8s) / (3s+1)
K_b = 1.8
tau_b = 3
theta_b = 0.8

Kp_b = 1.2 / (K_b * (theta_b / tau_b))
Ti_b = 2 * theta_b
Td_b = 0.5 * theta_b

print(f"Kp (b): {Kp_b:.4f}")
print(f"Ti (b): {Ti_b:.4f} s")
print(f"Td (b): {Td_b:.4f} s")

Irrigação automatizada
Kp (a): 2.4000
Ti (a): 8.0000 s
Td (a): 2.0000 s

Esteira transportadora industrial
Kp (b): 2.5000
Ti (b): 1.6000 s
Td (b): 0.4000 s


c) Comparação e análise de sensibilidade a atrasos adicionais

In [11]:
print("\n--- Comparação --- ")
print(f"Sistema (a) - Irrigação: Ti = {Ti_a:.4f} s, theta = {theta_a:.4f} s")
print(f"Sistema (b) - Esteira: Ti = {Ti_b:.4f} s, theta = {theta_b:.4f} s")

# Atraso adicional
delay_wifi = 0.3 # segundos

# Análise de sensibilidade:
# Um sistema é mais sensível a atrasos adicionais quando o tempo morto é uma porção significativa
# da sua dinâmica total (tau + theta) e, em particular, em relação ao Ti.
# Um Ti pequeno significa uma ação integrativa mais agressiva, que pode ser desestabilizante
# com um tempo morto elevado ou que aumenta ainda mais.

# Comparando Ti e theta
# Para o sistema (a): Ti_a = 8s, theta_a = 4s. Ratio theta/Ti = 4/8 = 0.5
# Para o sistema (b): Ti_b = 1.6s, theta_b = 0.8s. Ratio theta/Ti = 0.8/1.6 = 0.5


# Novo tempo morto com o atraso adicional
theta_a_new = theta_a + delay_wifi
theta_b_new = theta_b + delay_wifi

print(f"\nNovo theta (a) com atraso: {theta_a_new:.4f} s (era {theta_a:.4f} s)")
print(f"Novo theta (b) com atraso: {theta_b_new:.4f} s (era {theta_b:.4f} s)")

# Analisando o impacto relativo do atraso adicional
impact_a = (delay_wifi / theta_a) * 100
impact_b = (delay_wifi / theta_b) * 100

print(f"\nImpacto percentual do atraso de 300ms sobre theta (a): {impact_a:.2f}%")
print(f"Impacto percentual do atraso de 300ms sobre theta (b): {impact_b:.2f}%")

print("\n--- Justificativa ---")
if impact_a > impact_b:
    print(f"O sistema da Irrigação automatizada (a) seria **menos** sensível a um atraso adicional de 300ms. ")
else:
    print(f"O sistema da Esteira transportadora industrial (b) seria **mais** sensível a um atraso adicional de 300ms. ")



--- Comparação --- 
Sistema (a) - Irrigação: Ti = 8.0000 s, theta = 4.0000 s
Sistema (b) - Esteira: Ti = 1.6000 s, theta = 0.8000 s

Novo theta (a) com atraso: 4.3000 s (era 4.0000 s)
Novo theta (b) com atraso: 1.1000 s (era 0.8000 s)

Impacto percentual do atraso de 300ms sobre theta (a): 7.50%
Impacto percentual do atraso de 300ms sobre theta (b): 37.50%

--- Justificativa ---
O sistema da Esteira transportadora industrial (b) seria **mais** sensível a um atraso adicional de 300ms. 


**Resposta**

O sistema da Esteira transportadora industrial (b) seria mais sensível a um atraso adicional de 300ms. Isso se deve ao fato de que o tempo morto original (theta_b = 0.8s) é menor, fazendo com que o atraso adicional de 300ms represente um impacto percentual muito maior (37.50%) no tempo morto total do que para o sistema de Irrigação (7.50%). Um aumento relativo maior no tempo morto, combinado com um Ti menor, indica uma maior probabilidade de desestabilização do controle.

#Exercício 4

Um armário de rede (data center IoT) usa um cooler PWM e um sensor DHT22
para manter a temperatura interna. G(s) = 1,5e^(−1s) / (6s+1).

a) Calcule Kp, Ti e Td do controlador PID pelo método CHR sem sobrevalor.

b) Calcule Kp, Ti e Td do controlador PID pelo método CHR com 20% de
sobrevalor.

c) Plotar os gráficos de resposta a um degrau de valor 15.

d) Compare os dois Kp obtidos. Para um data center, ultrapassar a temperatura alvo mesmo que temporariamente pode acelerar o desgaste dos equipamentos.
Qual dos dois ajustes você recomendaria para este projeto e por quê?

In [15]:
# Parâmetros da planta para o Exercício 4:
# G(s) = 1,5e^(−1s) / (6s+1)
K_plant_ex4 = 1.5
tau_plant_ex4 = 6
theta_plant_ex4 = 1

print("CHR sem sobrevalor (PID)")
# Fórmulas CHR sem sobrevalor para PID:
# Kp = 0.95/K * (tau/theta)
# Ti = 2.4 * theta
# Td = 0.4 * theta

Kp_CHR_no_overshoot = 0.95 / K_plant_ex4 * (tau_plant_ex4 / theta_plant_ex4)
Ti_CHR_no_overshoot = 2.4 * theta_plant_ex4
Td_CHR_no_overshoot = 0.4 * theta_plant_ex4

print(f"Kp (CHR sem sobrevalor): {Kp_CHR_no_overshoot:.4f}")
print(f"Ti (CHR sem sobrevalor): {Ti_CHR_no_overshoot:.4f} s")
print(f"Td (CHR sem sobrevalor): {Td_CHR_no_overshoot:.4f} s\n")

print("CHR com 20% de sobrevalor (PID)")
# Fórmulas CHR com 20% de sobrevalor para PID:
# Kp = 1.2/K * (tau/theta)
# Ti = 2 * theta
# Td = 0.5 * theta

Kp_CHR_20_overshoot = 1.2 / K_plant_ex4 * (tau_plant_ex4 / theta_plant_ex4)
Ti_CHR_20_overshoot = 2 * theta_plant_ex4
Td_CHR_20_overshoot = 0.5 * theta_plant_ex4

print(f"Kp (CHR com 20% de sobrevalor): {Kp_CHR_20_overshoot:.4f}")
print(f"Ti (CHR com 20% de sobrevalor): {Ti_CHR_20_overshoot:.4f} s")
print(f"Td (CHR com 20% de sobrevalor): {Td_CHR_20_overshoot:.4f} s")

CHR sem sobrevalor (PID)
Kp (CHR sem sobrevalor): 3.8000
Ti (CHR sem sobrevalor): 2.4000 s
Td (CHR sem sobrevalor): 0.4000 s

CHR com 20% de sobrevalor (PID)
Kp (CHR com 20% de sobrevalor): 4.8000
Ti (CHR com 20% de sobrevalor): 2.0000 s
Td (CHR com 20% de sobrevalor): 0.5000 s


In [16]:
# Parâmetros da planta para o Exercício 4:
# G(s) = 1,5e^(−1s) / (6s+1)
K_plant_ex4 = 1.5
tau_plant_ex4 = 6
theta_plant_ex4 = 1

# Valor do degrau de entrada
step_value = 15

# Função para simular e plotar a resposta ao degrau
def plot_step_response_ex4(Kp, Ti, Td, method_name, color, fig, step_magnitude):
    # Função de Transferência da Planta (FOPDT): G(s) = K * e^(-theta*s) / (tau*s + 1)
    num_plant = np.array([K_plant_ex4])
    den_plant = np.array([tau_plant_ex4, 1])
    H_plant = cnt.tf(num_plant, den_plant)

    # Aproximação de Padé para o tempo morto
    n_pade = 10 # Ordem da aproximação de Padé
    (num_pade, den_pade) = cnt.pade(theta_plant_ex4, n_pade)
    H_pade = cnt.tf(num_pade, den_pade)

    # Controlador PID
    # Kp
    num_kp = np.array([Kp])
    den_kp = np.array([1])
    H_kp = cnt.tf(num_kp, den_kp)

    # Ki = Kp/Ti
    if Ti != 0:
        num_ki = np.array([Kp])
        den_ki = np.array([Ti, 0])
        H_ki = cnt.tf(num_ki, den_ki)
    else:
        H_ki = cnt.tf([0], [1]) # Termo integrativo zero se Ti for zero

    # Kd = Kp*Td
    num_kd = np.array([Kp * Td, 0])
    den_kd = np.array([1])
    H_kd = cnt.tf(num_kd, den_kd)

    # Controlador PID em paralelo
    H_ctrl_pi = cnt.parallel(H_kp, H_ki)
    H_ctrl_pid = cnt.parallel(H_ctrl_pi, H_kd)

    # Malha aberta: Planta + Atraso + Controlador
    H_open_loop_plant_delay = cnt.series(H_plant, H_pade)
    H_open_loop = cnt.series(H_open_loop_plant_delay, H_ctrl_pid)

    # Malha fechada
    H_cl = cnt.feedback(H_open_loop, 1)

    # Simulação da resposta ao degrau
    t = np.linspace(0, 40, 500) # Ajustar o tempo de simulação conforme necessário
    (t, y) = cnt.step_response(H_cl, t)

    # Adiciona a resposta ao gráfico, multiplicando pela magnitude do degrau
    fig.add_trace(go.Scatter(x=t, y=y * step_magnitude, mode='lines', name=method_name, line=dict(color=color)))

    return fig

# Inicializa a figura Plotly
fig_ex4 = go.Figure()

# Plotar para CHR sem sobrevalor
fig_ex4 = plot_step_response_ex4(Kp_CHR_no_overshoot, Ti_CHR_no_overshoot, Td_CHR_no_overshoot, 'CHR sem sobrevalor', 'purple', fig_ex4, step_value)

# Plotar para CHR com 20% de sobrevalor
fig_ex4 = plot_step_response_ex4(Kp_CHR_20_overshoot, Ti_CHR_20_overshoot, Td_CHR_20_overshoot, 'CHR com 20% de sobrevalor', 'orange', fig_ex4, step_value)

# Ajustes do layout do gráfico
fig_ex4.update_layout(
    title=f"Respostas ao Degrau (valor {step_value}) para CHR (Data Center)",
    xaxis_title="Tempo [s]",
    yaxis_title="Saída da Planta (°C)",
    width=900,
    height=600,
    template="plotly_white"
)

# Mostra o gráfico
fig_ex4.show()

d) **Resposta:**

O mais recomendado é o ajuste CHR sem sobrevalor (Kp = 3.8000, Ti = 2.4000 s, Td = 0.4000 s) para este projeto. Embora possa resultar em uma resposta um pouco mais lenta para atingir a temperatura alvo, a principal vantagem é que ele foi projetado para não apresentar sobrevalor. Isso garante que a temperatura no data center não ultrapasse o limite seguro, protegendo os equipamentos e mantendo a operação confiável. Em um data center, a estabilidade e a prevenção de picos de temperatura são mais críticas do que a velocidade de resposta extrema.

#Exercício 5

Usando o mesmo sistema do Bloco 0 (estufa agrícola, K = 60, τ = 40,5 s, θ = 1,5
s):

a) Calcule Kp, Ti e Td do PID por IMC usando λ = 1,5θ.

b) Calcule Kp, Ti e Td do PID por IMC usando λ = 4θ.

c) Plotar os gráficos de resposta a um degrau de valor 20.

d) Explique, com base nos dois resultados, a relação entre λ, velocidade de
resposta e robustez a perturbações. Se a estufa estiver sujeita a rajadas de vento
que abrem e fecham as telas de ventilação com frequência, qual λ você escolheria?

In [19]:
# Parâmetros do sistema da estufa agrícola (Bloco 0):
K_plant_ex5 = 60
tau_plant_ex5 = 40.5
theta_plant_ex5 = 1.5


print("IMC usando λ = 1.5θ")
lambda_a = 1.5 * theta_plant_ex5

Ti_IMC_a = tau_plant_ex5 + theta_plant_ex5 / 2
Td_IMC_a = (tau_plant_ex5 * theta_plant_ex5) / (2 * tau_plant_ex5 + theta_plant_ex5)
Kp_IMC_a = (1 / K_plant_ex5) * (Ti_IMC_a) / (lambda_a + theta_plant_ex5 / 2)

print(f"Lambda (a): {lambda_a:.4f} s")
print(f"Kp (IMC, λ=1.5θ): {Kp_IMC_a:.4f}")
print(f"Ti (IMC, λ=1.5θ): {Ti_IMC_a:.4f} s")
print(f"Td (IMC, λ=1.5θ): {Td_IMC_a:.4f} s\n")

print("IMC usando λ = 4θ")
lambda_b = 4 * theta_plant_ex5

Ti_IMC_b = tau_plant_ex5 + theta_plant_ex5 / 2 # Ti e Td são os mesmos, pois não dependem de lambda
Td_IMC_b = (tau_plant_ex5 * theta_plant_ex5) / (2 * tau_plant_ex5 + theta_plant_ex5)
Kp_IMC_b = (1 / K_plant_ex5) * (Ti_IMC_b) / (lambda_b + theta_plant_ex5 / 2)

print(f"Lambda (b): {lambda_b:.4f} s")
print(f"Kp (IMC, λ=4θ): {Kp_IMC_b:.4f}")
print(f"Ti (IMC, λ=4θ): {Ti_IMC_b:.4f} s")
print(f"Td (IMC, λ=4θ): {Td_IMC_b:.4f} s")

IMC usando λ = 1.5θ
Lambda (a): 2.2500 s
Kp (IMC, λ=1.5θ): 0.2292
Ti (IMC, λ=1.5θ): 41.2500 s
Td (IMC, λ=1.5θ): 0.7364 s

IMC usando λ = 4θ
Lambda (b): 6.0000 s
Kp (IMC, λ=4θ): 0.1019
Ti (IMC, λ=4θ): 41.2500 s
Td (IMC, λ=4θ): 0.7364 s


c) Plotagem das respostas ao degrau (degrau de valor 20) para os controladores IMC

In [18]:
# Parâmetros da planta identificados (Exercício 5, Bloco 0)
k_plant_ex5 = K_plant_ex5 # Usando o K da estufa agrícola
tau_plant_ex5 = tau_plant_ex5 # Usando o tau da estufa agrícola
theta_plant_ex5 = theta_plant_ex5 # Usando o theta da estufa agrícola

# Valor do degrau de entrada
step_value_ex5 = 20

# Função para simular e plotar a resposta ao degrau (adaptada do Exercício 1 e 4)
def plot_step_response_ex5(Kp, Ti, Td, method_name, color, fig, step_magnitude):
    # Função de Transferência da Planta (FOPDT): G(s) = K * e^(-theta*s) / (tau*s + 1)
    num_plant = np.array([k_plant_ex5])
    den_plant = np.array([tau_plant_ex5, 1])
    H_plant = cnt.tf(num_plant, den_plant)

    # Aproximação de Padé para o tempo morto
    n_pade = 10 # Ordem da aproximação de Padé
    (num_pade, den_pade) = cnt.pade(theta_plant_ex5, n_pade)
    H_pade = cnt.tf(num_pade, den_pade)

    # Controlador PID
    # Kp
    num_kp = np.array([Kp])
    den_kp = np.array([1])
    H_kp = cnt.tf(num_kp, den_kp)

    # Ki = Kp/Ti
    if Ti != 0:
        num_ki = np.array([Kp])
        den_ki = np.array([Ti, 0])
        H_ki = cnt.tf(num_ki, den_ki)
    else:
        H_ki = cnt.tf([0], [1]) # Termo integrativo zero se Ti for zero

    # Kd = Kp*Td
    num_kd = np.array([Kp * Td, 0])
    den_kd = np.array([1])
    H_kd = cnt.tf(num_kd, den_kd)

    # Controlador PID em paralelo
    H_ctrl_pi = cnt.parallel(H_kp, H_ki)
    H_ctrl_pid = cnt.parallel(H_ctrl_pi, H_kd)

    # Malha aberta: Planta + Atraso + Controlador
    H_open_loop_plant_delay = cnt.series(H_plant, H_pade)
    H_open_loop = cnt.series(H_open_loop_plant_delay, H_ctrl_pid)

    # Malha fechada
    H_cl = cnt.feedback(H_open_loop, 1)

    # Simulação da resposta ao degrau
    t = np.linspace(0, 300, 500) # Ajustar o tempo de simulação conforme necessário para ver a estabilização
    (t, y) = cnt.step_response(H_cl, t)

    # Adiciona a resposta ao gráfico, multiplicando pela magnitude do degrau
    fig.add_trace(go.Scatter(x=t, y=y * step_magnitude, mode='lines', name=method_name, line=dict(color=color)))

    return fig

# Inicializa a figura Plotly
fig_ex5 = go.Figure()

# Plotar para IMC com λ = 1.5θ
fig_ex5 = plot_step_response_ex5(Kp_IMC_a, Ti_IMC_a, Td_IMC_a, 'IMC (λ=1.5θ)', 'blue', fig_ex5, step_value_ex5)

# Plotar para IMC com λ = 4θ
fig_ex5 = plot_step_response_ex5(Kp_IMC_b, Ti_IMC_b, Td_IMC_b, 'IMC (λ=4θ)', 'red', fig_ex5, step_value_ex5)

# Ajustes do layout do gráfico
fig_ex5.update_layout(
    title=f"Respostas ao Degrau (valor {step_value_ex5}) para Controladores IMC na Estufa Agrícola",
    xaxis_title="Tempo [s]",
    yaxis_title="Saída da Planta (°C)",
    width=900,
    height=600,
    template="plotly_white"
)

# Mostra o gráfico
fig_ex5.show()

d) Relação entre λ, velocidade de resposta e robustez; recomendação para estufa com rajadas de vento

**Análise da relação entre λ, velocidade de resposta e robustez:**

Nos métodos de sintonia IMC, o parâmetro $\lambda$ é um fator de ajuste que permite equilibrar o desempenho do controlador em termos de velocidade de resposta e robustez a incertezas do modelo ou perturbações:

*   **$\lambda$ pequeno (e.g., $\lambda = 1.5\theta$):** Um valor menor de $\lambda$ tende a resultar em um controlador mais agressivo, com um ganho proporcional (Kp) maior. Isso geralmente leva a uma **resposta mais rápida** a mudanças no ponto de ajuste ou a perturbações, atingindo o valor desejado mais rapidamente. No entanto, um $\lambda$ pequeno implica em menor margem de robustez. Isso significa que o controlador é **menos robusto** a erros no modelo da planta (se os parâmetros K, $\tau$, $\theta$ não forem perfeitamente precisos) ou a perturbações não modeladas, podendo levar a oscilações ou instabilidade.

*   **$\lambda$ grande (e.g., $\lambda = 4\theta$):** Um valor maior de $\lambda$ leva a um controlador mais conservador, com um ganho proporcional (Kp) menor. Isso resulta em uma **resposta mais lenta**, com o sistema demorando mais para atingir o ponto de ajuste. Contudo, um $\lambda$ maior proporciona **maior robustez**. O controlador é mais tolerante a variações nos parâmetros da planta, a ruídos de medição e a perturbações externas, sendo menos propenso a oscilações e mais estável sob condições operacionais incertas.

Em resumo, $\lambda$ e a velocidade de resposta são inversamente proporcionais ($\downarrow \lambda \implies \uparrow$ velocidade), enquanto $\lambda$ e a robustez são diretamente proporcionais ($\uparrow \lambda \implies \uparrow$ robustez). Escolher um $\lambda$ é um compromisso entre esses dois aspectos.

**Recomendação para a estufa sujeita a rajadas de vento:**

No cenário de uma estufa sujeita a rajadas de vento que abrem e fecham as telas de ventilação frequentemente, estamos lidando com **perturbações externas significativas e frequentes**. Essas perturbações podem ser vistas como variações rápidas e imprevisíveis na demanda de calor ou na troca térmica da estufa, ou mesmo como incertezas no modelo da planta devido a essas variações. Nessas condições, a **robustez** do controlador se torna um fator crucial para manter a temperatura estável e evitar grandes oscilações.

Portanto, eu recomendaria escolher o valor de **$\lambda = 4\theta$** (que resultou em Kp = 0.1019). Embora esta configuração ofereça uma resposta mais lenta a um degrau (como visto no gráfico, a curva vermelha é mais suave e mais lenta para atingir o valor final), ela garante uma **maior robustez** contra as perturbações causadas pelas rajadas de vento. Um controlador mais robusto será mais eficaz em amortecer o impacto dessas perturbações, mantendo a temperatura interna da estufa mais estável e consistente, o que é fundamental para o cultivo agrícola.